<a href="https://colab.research.google.com/github/abdus-git/NLP/blob/main/unit2/23Attention_Mechanism.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

```markdown
## Basic Custom Attention Mechanism

We will implement a simple scaled dot-product attention mechanism (Self-Attention) using PyTorch.
$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V$$
```

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class SimpleSelfAttention(nn.Module):
    def __init__(self, embed_dim):
        super(SimpleSelfAttention, self).__init__()
        self.embed_dim = embed_dim

        # Linear projections for Query, Key, and Value
        self.q_linear = nn.Linear(embed_dim, embed_dim)
        self.k_linear = nn.Linear(embed_dim, embed_dim)
        self.v_linear = nn.Linear(embed_dim, embed_dim)

        # Final output projection
        self.out_linear = nn.Linear(embed_dim, embed_dim)

    def forward(self, x):
        # x shape: [batch_size, seq_len, embed_dim]
        batch_size, seq_len, embed_dim = x.size()

        # 1. Project to Q, K, V
        Q = self.q_linear(x)  # [batch_size, seq_len, embed_dim]
        K = self.k_linear(x)  # [batch_size, seq_len, embed_dim]
        V = self.v_linear(x)  # [batch_size, seq_len, embed_dim]

        # 2. Calculate attention scores (dot product of Q and K^T)
        # We transpose the last two dimensions of K to do batch matrix multiplication
        scores = torch.bmm(Q, K.transpose(1, 2))  # [batch_size, seq_len, seq_len]

        # 3. Scale the scores
        scores = scores / (embed_dim ** 0.5)

        # 4. Apply softmax to get attention weights
        attention_weights = F.softmax(scores, dim=-1)  # [batch_size, seq_len, seq_len]

        # 5. Multiply weights by V
        out = torch.bmm(attention_weights, V)  # [batch_size, seq_len, embed_dim]

        # 6. Apply final linear layer
        out = self.out_linear(out)

        return out, attention_weights

```markdown
### Test and Verify the Attention Model
Let's test our module with some dummy dimensions.
```

In [2]:
# Define dimensions
batch_size = 2
seq_len = 5
embed_dim = 16

# Create model
attention_layer = SimpleSelfAttention(embed_dim=embed_dim)

# Generate random input tensor: [batch, seq_len, embed_dim]
x = torch.randn(batch_size, seq_len, embed_dim)

# Run forward pass
output, weights = attention_layer(x)

print("Input shape:", x.shape)
print("Output shape:", output.shape)
print("Attention Weights shape:", weights.shape)
print("\nExample Attention Matrix for Batch 0:\n", weights[0].detach().numpy())

Input shape: torch.Size([2, 5, 16])
Output shape: torch.Size([2, 5, 16])
Attention Weights shape: torch.Size([2, 5, 5])

Example Attention Matrix for Batch 0:
 [[0.13528939 0.13613757 0.18529938 0.25904298 0.28423068]
 [0.1558914  0.18665247 0.16877908 0.30148244 0.18719459]
 [0.22936527 0.16927849 0.25609156 0.1822648  0.16299987]
 [0.2024537  0.09406964 0.32634753 0.1978153  0.17931384]
 [0.26328355 0.10858209 0.26553872 0.19390893 0.16868676]]
